# Notebook 1 — Guide Python / NumPy / SciPy

Format : formule mathématique générique, code Python, description de la fonction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.signal import lfilter, freqz, tf2zpk
from scipy.linalg import toeplitz, solve, inv, pinv

# 1. Création de tableaux

## 1.1 Vecteur / matrice à partir de valeurs

$$\mathbf{x} = [x_0, x_1, \dots, x_{N-1}]^\top$$

In [ ]:
x = np.array([1.0, 2.0, 3.0])
A = np.array([[1, 2], [3, 4]])

- `np.array(obj)` : convertit une liste (ou liste de listes) en `ndarray`.
- `obj` : structure imbriquée Python. Le nombre de niveaux d'imbrication détermine le nombre de dimensions.
- retour : `ndarray` de shape correspondante.

## 1.2 Vecteur de zéros / de uns

$$\mathbf{0}_N = [0, \dots, 0], \qquad \mathbf{1}_{M \times N} = \text{matrice de 1}$$

In [ ]:
z = np.zeros(N)
o = np.ones((M, N))

- `np.zeros(shape)` / `np.ones(shape)` : remplit de 0 ou 1.
- `shape` : entier `N` → vecteur 1D de taille `N` ; tuple `(M, N)` → matrice 2D.
- retour : `ndarray` rempli, dtype `float64` par défaut.

## 1.3 Matrice identité

$$\mathbf{I}_N \in \mathbb{R}^{N \times N}, \quad I_{ij} = \delta_{ij}$$

In [ ]:
I = np.eye(N)

- `np.eye(N, M=None)` : matrice identité.
- `N` : nombre de lignes. `M` : nombre de colonnes (par défaut `M=N`).
- retour : matrice `(N, M)` avec des 1 sur la diagonale principale.

## 1.4 Impulsion de Dirac discrète

$$\boldsymbol{\delta} = [1, 0, 0, \dots, 0]$$

In [ ]:
dn = np.eye(1, N)[0]

- `np.eye(1, N)` : matrice `(1, N)` avec un 1 en position `(0, 0)`.
- `[0]` : extrait la première (et unique) ligne → vecteur 1D de longueur `N` valant `[1, 0, ..., 0]`.

## 1.5 Plage d'entiers

$$n = 0, 1, 2, \dots, N-1$$

In [ ]:
n = np.arange(N)

- `np.arange(start, stop, step)` : équivalent vectorisé de `range`. Avec un seul argument, `start=0`, `step=1`.
- retour : vecteur 1D d'entiers, `stop` exclu.

## 1.6 Points uniformément répartis sur un intervalle

$$f_k = a + k\frac{b-a}{K-1}, \quad k = 0, \dots, K-1$$

In [ ]:
f = np.linspace(a, b, K)

- `np.linspace(start, stop, num)` : `num` points équidistants entre `start` et `stop`.
- `stop` est **inclus** (contrairement à `arange`).
- retour : vecteur 1D de longueur `num`.

# 2. Manipulation de shape

## 2.1 Shape d'un tableau

In [ ]:
x.shape

- Attribut, pas une méthode (pas de parenthèses).
- retour : tuple. Vecteur 1D → `(N,)`. Matrice 2D → `(M, N)`.

## 2.2 Reshape en ligne / colonne

Vecteur ligne $\in \mathbb{R}^{1 \times N}$, vecteur colonne $\in \mathbb{R}^{N \times 1}$.

In [ ]:
x.reshape(-1, 1)       # colonne (N, 1)
x.reshape(1, -1)       # ligne   (1, N)
x[:, None]             # raccourci colonne

- `arr.reshape(*new_shape)` : change la shape sans copier les données. Le produit des dimensions doit rester constant.
- `-1` : NumPy déduit la dimension restante.
- `x[:, None]` : insère un nouvel axe → passe de `(N,)` à `(N, 1)`.

## 2.3 Transposée

$$(A^\top)_{ij} = A_{ji}$$

In [ ]:
A.T

- Attribut. Inverse l'ordre des axes.
- Sur un vecteur 1D `(N,)` : **n'a aucun effet** (il faut passer en 2D avant).

## 2.4 Concaténation

$$[\mathbf{a}, \mathbf{b}] = [a_0, \dots, a_{N-1}, b_0, \dots, b_{M-1}]$$

In [ ]:
c = np.concatenate([a, b])

- `np.concatenate(seq, axis=0)` : recolle des tableaux le long d'un axe.
- `seq` : liste/tuple de tableaux à concaténer. Toutes les dimensions sauf `axis` doivent être identiques.
- Usage fréquent : ajouter un coefficient en tête, par exemple `np.concatenate(([1.0], ap_tail))`.

# 3. Indexation et slicing

## 3.1 Élément, ligne, colonne, sous-matrice

$A_{ij}$, $\mathbf{A}_{i,:}$, $\mathbf{A}_{:,j}$.

In [ ]:
A[i, j]              # scalaire
A[i, :]              # ligne i
A[:, j]              # colonne j
A[a:b, c:d]          # sous-matrice, b et d exclus

- Slicing standard : `start:stop:step`, `stop` exclu, valeurs par défaut `0 : len : 1`.
- Pas de copie : modifier la vue modifie l'original (utiliser `.copy()` si besoin).

## 3.2 Indexation booléenne

Sélection des éléments où une condition est vraie.

In [ ]:
x[x > 0]

- `x > 0` : produit un tableau booléen de même shape que `x`.
- `x[mask]` : retourne les éléments où `mask` vaut `True`, **aplati en 1D**.

## 3.3 Inversion d'un vecteur

$$\tilde{x}(n) = x(N-1-n)$$

In [ ]:
x[::-1]

- Slicing avec `step = -1` : parcourt le tableau à l'envers.
- Retourne une vue (pas de copie).

# 4. Opérations

## 4.1 Opérations élément par élément (Hadamard)

$$(A \odot B)_{ij} = A_{ij} \cdot B_{ij}$$

In [ ]:
A * B
A ** 2

- `*`, `/`, `+`, `-`, `**` : toujours élément par élément en NumPy.
- Les opérandes doivent avoir la même shape, ou être compatibles par broadcasting.

## 4.2 Produit matriciel

$$(AB)_{ij} = \sum_k A_{ik}\, B_{kj}$$

In [ ]:
A @ B

- `@` : produit matriciel (Python 3.5+). Équivalent à `np.matmul(A, B)`.
- Vecteur 1D `@` vecteur 1D → scalaire (produit scalaire).
- Vérifier les shapes : `(M, K) @ (K, N) → (M, N)`.

## 4.3 Broadcasting

Une dimension de taille 1 (ou absente) est étirée pour matcher.

$$\tilde{X}_{n,d} = X_{n,d} - \mu_d$$

In [ ]:
X - mu              # (N, D) - (D,) → (N, D)
X - mu[:, None]     # (N, D) - (N, 1) → broadcast sur les colonnes

- Règle : NumPy aligne les shapes par la droite. Dimension manquante = 1.
- Une dimension de taille 1 est répétée pour matcher l'autre opérande.

## 4.4 Réductions

Somme, moyenne, max, etc. le long d'un axe.

$$\bar{x}_d = \frac{1}{N}\sum_{n=0}^{N-1} X_{n,d}$$

In [ ]:
X.sum(axis=0)
X.mean(axis=0)
X.max(axis=1, keepdims=True)
np.argmax(X, axis=1)

- `axis=0` : réduit le long des lignes → résultat de shape `(D,)` (stat par colonne).
- `axis=1` : réduit le long des colonnes → shape `(N,)` (stat par ligne).
- `axis=None` (par défaut) : réduit sur tous les axes → scalaire.
- `keepdims=True` : garde la dimension réduite à taille 1 (utile pour broadcasting ultérieur).
- `argmax`, `argmin` : retournent l'**indice** du maximum/minimum.

## 4.5 Norme L2

$$\|\mathbf{x}\|_2 = \sqrt{\sum_n x_n^2}$$

In [ ]:
np.linalg.norm(x)

- `np.linalg.norm(x, ord=None, axis=None)`.
- `ord=None` : norme 2 (euclidienne). `ord=1` : somme des valeurs absolues. `ord=np.inf` : max des absolus.
- `axis` : permet de calculer la norme par ligne ou par colonne.

# 5. Aléatoire

## 5.1 Initialisation d'un générateur

Reproductibilité : même graine → même séquence.

In [ ]:
rng = np.random.default_rng(seed)
np.random.seed(seed)

- `default_rng(seed)` : API moderne, retourne un objet générateur isolé.
- `np.random.seed(seed)` : API legacy, agit sur un état global partagé.
- `seed` : entier (n'importe lequel, souvent 0 ou 42).

## 5.2 Loi uniforme

$$X \sim \mathcal{U}(a, b)$$

In [ ]:
rng.uniform(low, high, size)

- `low`, `high` : bornes (incluses pour low, exclues pour high).
- `size` : entier ou tuple → shape du tableau retourné.
- retour : `ndarray` de la shape demandée.

## 5.3 Loi normale

$$X \sim \mathcal{N}(\mu, \sigma^2)$$

In [ ]:
rng.normal(loc, scale, size)

- `loc` : moyenne $\mu$.
- `scale` : **écart-type** $\sigma$ (pas la variance).
- `size` : entier ou tuple.
- Pour un bruit blanc gaussien de variance $\sigma^2$ : `sigma * rng.normal(0, 1, size)`.

## 5.4 Entiers aléatoires

In [ ]:
rng.integers(low, high, size)

- `low` (inclus), `high` (exclu).
- retour : entiers tirés uniformément dans `[low, high)`.

# 6. Statistiques

## 6.1 Moyenne

$$\bar{x} = \frac{1}{N}\sum_n x_n$$

In [ ]:
np.mean(x, axis=None)

- `axis` : axe sur lequel calculer. `None` → scalaire global.

## 6.2 Variance

$$\sigma^2 = \frac{1}{N - \text{ddof}}\sum_n (x_n - \bar{x})^2$$

In [ ]:
np.var(x, axis=None, ddof=0)

- `ddof=0` (défaut) : diviseur $N$, estimateur **biaisé** (ML).
- `ddof=1` : diviseur $N-1$, estimateur **non biaisé** (échantillon).

## 6.3 Écart-type

$$\sigma = \sqrt{\sigma^2}$$

In [ ]:
np.std(x, axis=None, ddof=0)

- Même signature que `np.var`. Même remarque sur `ddof`.

## 6.4 Corrélation linéaire entre deux signaux

$$r_{xy} = \frac{\sum_n (x_n - \bar{x})(y_n - \bar{y})}{\sqrt{\sum_n (x_n - \bar{x})^2 \sum_n (y_n - \bar{y})^2}}$$

In [ ]:
np.corrcoef(x, y)

- retour : matrice $2 \times 2$ symétrique. Coefficient désiré : `[0, 1]` ou `[1, 0]`.
- Diagonale = 1 par construction.

# 7. Algèbre linéaire

## 7.1 Résolution de système linéaire

$$\mathbf{A}\mathbf{x} = \mathbf{b} \quad \Rightarrow \quad \mathbf{x} = \mathbf{A}^{-1}\mathbf{b}$$

In [ ]:
x = solve(A, b)

- `scipy.linalg.solve(A, b)` (ou `np.linalg.solve`) : résout par factorisation LU.
- `A` : matrice carrée `(n, n)`, inversible.
- `b` : vecteur `(n,)` ou matrice `(n, k)` pour plusieurs seconds membres.
- Plus stable et plus rapide que `inv(A) @ b`.

## 7.2 Inverse

$$\mathbf{A}^{-1}$$

In [ ]:
inv(A)

- `scipy.linalg.inv(A)` (ou `np.linalg.inv`).
- `A` : carrée et inversible (sinon `LinAlgError`).
- À éviter pour résoudre un système ; préférer `solve`.

## 7.3 Pseudo-inverse (Moore-Penrose)

$$\mathbf{A}^+ = (\mathbf{A}^\top \mathbf{A})^{-1}\mathbf{A}^\top \quad \text{(si } \mathbf{A} \text{ de rang plein en colonnes)}$$

In [ ]:
pinv(A)

- `scipy.linalg.pinv(A)` (ou `np.linalg.pinv`).
- Fonctionne pour matrices rectangulaires ou rang déficient.
- `pinv(A) @ b` ⇔ solution moindres carrés de $\mathbf{A}\mathbf{x} = \mathbf{b}$.

## 7.4 Moindres carrés

$$\min_{\mathbf{x}} \|\mathbf{A}\mathbf{x} - \mathbf{b}\|_2^2$$

In [ ]:
x, residuals, rank, sv = np.linalg.lstsq(A, b, rcond=None)

- Plus stable numériquement que `pinv(A) @ b`.
- `rcond=None` : utilise le seuil par défaut pour la troncature des petites valeurs singulières.
- retour : solution `x`, somme des carrés des résidus, rang de `A`, valeurs singulières.

## 7.5 Déterminant, trace, rang

In [ ]:
np.linalg.det(A)
np.trace(A)
np.linalg.matrix_rank(A)

- `det` : retourne un scalaire. Matrice singulière ⇔ déterminant nul.
- `trace` : somme de la diagonale principale.
- `matrix_rank` : nombre de valeurs singulières non négligeables.

## 7.6 Valeurs et vecteurs propres

$$\mathbf{A}\mathbf{v}_i = \lambda_i \mathbf{v}_i$$

In [ ]:
vals, vecs = np.linalg.eig(A)
vals, vecs = np.linalg.eigh(A)

- `eig` : cas général. `vals` peut être complexe.
- `eigh` : matrice symétrique/hermitienne. Plus rapide et stable, `vals` réelles.
- Les vecteurs propres sont **les colonnes** de `vecs`, pas les lignes.

## 7.7 Racines d'un polynôme

Racines de $p(z) = c_0 z^n + c_1 z^{n-1} + \dots + c_n$.

In [ ]:
np.roots(c)

- `c` : vecteur des coefficients, **du plus haut au plus bas degré**.
- retour : vecteur complexe des $n$ racines.
- Pour $A(z) = 1 + a_1 z^{-1} + \dots + a_p z^{-p}$, passer `[1, a1, ..., ap]` → racines = pôles du filtre $1/A(z)$.

## 7.8 Module, argument, partie réelle/imaginaire d'un complexe

In [ ]:
np.abs(z)
np.angle(z, deg=False)
np.real(z)
np.imag(z)

- `np.abs` : module $|z|$.
- `np.angle` : argument en radians ; `deg=True` pour des degrés.
- `np.real`, `np.imag` : parties réelle et imaginaire.

## 7.9 Diagonale d'une matrice

In [ ]:
np.diag(M, k=0)
np.diag(v)

- Sur une matrice `M` : retourne la `k`-ième diagonale en vecteur 1D. `k=0` principale, `k>0` au-dessus, `k<0` en-dessous.
- Sur un vecteur `v` : retourne une matrice diagonale de taille `(len(v), len(v))` avec `v` sur la diagonale.

## 7.10 Matrice de Toeplitz

Matrice à diagonales constantes : $T_{ij} = t_{i-j}$.

In [ ]:
T = toeplitz(col, row)
T = toeplitz(r)

- `scipy.linalg.toeplitz(c, r=None)`.
- `c` : première colonne. `r` : première ligne (par défaut `r = c` → matrice symétrique).
- L'élément `c[0]` doit valoir `r[0]` (sinon `c[0]` prévaut).
- Shape retournée : `(len(c), len(r))`.

## 7.11 Matrice de Vandermonde

$$V_{n,i} = z_i^n$$

In [ ]:
V = np.vander(z, N, increasing=True)

- `z` : vecteur des bases (longueur `p`).
- `N` : nombre de puissances (lignes du résultat).
- `increasing=True` : puissances croissantes $z^0, z^1, \dots, z^{N-1}$. Par défaut `False` (décroissant).
- Shape retournée : `(len(z), N)`. Transposer si on veut $N$ lignes (échantillons) et $p$ colonnes (pôles).

# 8. Traitement du signal (scipy.signal)

## 8.1 Filtrage d'un signal

Filtre rationnel $H(z) = B(z)/A(z)$ :

$$\sum_{k=0}^{p} a_k\, y(n-k) = \sum_{k=0}^{q} b_k\, x(n-k), \quad a_0 = 1$$

In [ ]:
y = lfilter(b, a, x)

- `scipy.signal.lfilter(b, a, x)`.
- `b` : coefficients du numérateur, vecteur de longueur `q+1`.
- `a` : coefficients du dénominateur, vecteur de longueur `p+1`. **`a[0]` doit être non nul** (normalement 1).
- `x` : signal d'entrée.
- retour : signal filtré de même longueur que `x`.
- Filtre FIR : passer `a = [1]` ou `a = 1`.

## 8.2 Réponse impulsionnelle d'un filtre

$$h(n) = \mathcal{Z}^{-1}\{H(z)\}$$

Obtenue en filtrant une impulsion $\delta(n)$.

In [ ]:
hn = lfilter(b, a, np.eye(1, N)[0])

- Construction : on filtre l'impulsion de Dirac de longueur `N`.
- retour : vecteur de longueur `N` contenant les `N` premiers échantillons de $h(n)$.

## 8.3 Réponse en fréquence

$$H(e^{j\omega}) = \frac{B(e^{j\omega})}{A(e^{j\omega})}, \quad \omega \in [0, \pi]$$

In [ ]:
w, H = freqz(b, a, worN=1024)

- `scipy.signal.freqz(b, a, worN)`.
- `worN` : nombre de points de pulsation (entier) ou tableau de pulsations spécifiques.
- retour : `w` pulsations en rad/échantillon dans $[0, \pi]$, `H` valeurs complexes de $H(e^{j\omega})$.
- Fréquence normalisée : `f = w / (2*np.pi)` (intervalle $[0, 0.5]$).
- Magnitude en dB : `20*np.log10(np.abs(H))`. Phase : `np.angle(H)`.

## 8.4 Zéros, pôles, gain d'un filtre

$$H(z) = K \frac{\prod_i (z - z_i)}{\prod_j (z - p_j)}$$

In [ ]:
zeros, poles, k = tf2zpk(b, a)

- `scipy.signal.tf2zpk(b, a)` : Transfer Function → Zeros, Poles, gain.
- retour : `zeros` (vecteur complexe), `poles` (vecteur complexe), `k` (scalaire = gain).
- Stabilité : tous les `|poles| < 1`.

## 8.5 Corrélation entre deux signaux

$$r_{xy}(k) = \sum_n x(n)\, y(n+k)$$

In [ ]:
r = np.correlate(x, y, mode='full')

- `mode='full'` : retourne tous les lags, vecteur de longueur `len(x) + len(y) - 1`.
- `mode='valid'` : uniquement les lags où il n'y a pas de débordement.
- Lag 0 à l'indice `len(x) - 1` (pour `mode='full'` et `len(x) == len(y)`).
- Pour l'autocorrélation : `np.correlate(x, x, mode='full')`.
- Pour normaliser en estimateur biaisé : diviser par `N`.

## 8.6 Convolution

$$y(n) = (x * h)(n) = \sum_k x(k)\, h(n-k)$$

In [ ]:
y = np.convolve(x, h, mode='full')

- `mode='full'` : longueur `len(x) + len(h) - 1`.
- `mode='same'` : même longueur que `x`.
- `mode='valid'` : uniquement la partie sans débordement.

## 8.7 Sinus cardinal

$$\text{sinc}(x) = \frac{\sin(\pi x)}{\pi x}$$

In [ ]:
np.sinc(x)

- ⚠️ Convention **normalisée** : `np.sinc(x) = sin(π x) / (π x)`, pas `sin(x)/x`.
- `sinc(0) = 1` par continuité.

# 9. Affichage (matplotlib)

## 9.1 Tracé continu

In [ ]:
plt.plot(x, y, fmt, label='...')

- `x`, `y` : abscisses, ordonnées.
- `fmt` : chaîne combinant couleur + marqueur + style de ligne. Ex : `'b-'` bleu trait plein, `'r--'` rouge tirets, `'go'` cercles verts.
- `label` : utilisé par `plt.legend()`.

## 9.2 Tracé en bâtons (signal discret)

In [ ]:
plt.stem(n, x, linefmt='b-', markerfmt='bo')

- `linefmt` : style de la barre verticale.
- `markerfmt` : style du marqueur en haut.

## 9.3 Mise en forme courante

In [ ]:
plt.figure(figsize=(W, H))
plt.title('...')
plt.xlabel('...'); plt.ylabel('...')
plt.legend()
plt.grid(True)
plt.show()

## 9.4 Sous-graphiques

In [ ]:
plt.subplot(nrows, ncols, index)

- Index commence à **1**, parcourt les positions de gauche à droite, puis de haut en bas.
- Alternative orientée objet : `fig, axes = plt.subplots(nrows, ncols)`.